In [ ]:
# Bulk-pulls ~180 days of daily Near Mint price history for many cards from
# PokemonPriceTracker, and saves a clean time-series dataset for model training.
#
# Strategy: pull set by set (credit-efficient), extract just the Near Mint
# daily history from each card, save incrementally so a mid-run failure
# doesn't lose progress.

import requests
import json
import time

API_KEY = "userdata.get('PokeData')"
BASE_URL = "https://www.pokemonpricetracker.com/api/v2"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

TARGET_CONDITION = "Near Mint"
HISTORY_DAYS = 180
CARD_TARGET = 9000  # bumped up — with 20k credits (~2 each) this is safely affordable


def fetch_with_retry(url, params, max_retries=4):
    """Retry server errors with exponential backoff (same pattern as before)."""
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code >= 500:
                wait = 2 ** attempt
                print(f"  Got {r.status_code}, retrying in {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code == 429:
                print("  Rate limited (429) — waiting 60s before retrying...")
                time.sleep(60)
                continue
            r.raise_for_status()
            return r
        except requests.exceptions.Timeout:
            wait = 2 ** attempt
            print(f"  Timeout, retrying in {wait}s...")
            time.sleep(wait)
    return None


def get_all_sets():
    """Grab the list of all sets, so we can pull cards set by set."""
    r = fetch_with_retry(f"{BASE_URL}/sets", {"limit": 250})
    if r is None:
        return []
    return r.json().get("data", [])


def extract_nm_history(card):
    """Pulls out just the Near Mint daily price history from one card's
    full record, returning a clean list of {date, price, volume} points."""
    ph = card.get("priceHistory", {})
    conditions = ph.get("conditions", {})
    nm = conditions.get(TARGET_CONDITION, {})
    history = nm.get("history", [])

    cleaned = []
    for point in history:
        if point.get("market") is not None:
            cleaned.append({
                "date": point["date"][:10],  # just YYYY-MM-DD
                "price": point["market"],
                "volume": point.get("volume"),
            })
    return cleaned


def build_card_record(card):
    """Builds one training record: card metadata + its NM price time series."""
    return {
        "tcgPlayerId": card.get("tcgPlayerId"),
        "name": card.get("name"),
        "setName": card.get("setName"),
        "rarity": card.get("rarity"),
        "pokemonType": card.get("pokemonType"),
        "imageCdnUrl200": card.get("imageCdnUrl200"),
        "imageCdnUrl400": card.get("imageCdnUrl400"),
        "imageUrl": card.get("imageUrl"),
        "tcgPlayerUrl": card.get("tcgPlayerUrl"),
        "price_history": extract_nm_history(card),
    }


if __name__ == "__main__":
    sets = get_all_sets()
    print(f"Found {len(sets)} sets. Pulling cards with {HISTORY_DAYS} days of history...\n")

    all_records = []
    for i, s in enumerate(sets, start=1):
        # Skip sets with no cards yet (e.g. unreleased 2026 sets)
        if s.get("cardCount", 0) == 0:
            continue

        # The diagnostic proved the `set=` filter works with the set NAME
        # (e.g. "Base Set" returned 102 cards). tcgPlayerId did NOT match.
        set_name = s.get("name")
        set_filter = set_name

        params = {
            "set": set_filter,
            "fetchAllInSet": "true",
            "includeHistory": "true",
            "days": HISTORY_DAYS,
        }
        r = fetch_with_retry(f"{BASE_URL}/cards", params)
        if r is None:
            print(f"[{i}/{len(sets)}] {set_name}: FAILED, skipping")
            continue

        cards = r.json().get("data", [])
        # keep only cards that actually have Near Mint history
        records = [build_card_record(c) for c in cards]
        records = [rec for rec in records if rec["price_history"]]
        all_records.extend(records)

        remaining = r.headers.get("X-RateLimit-Daily-Remaining", "?")
        print(f"[{i}/{len(sets)}] {set_name}: +{len(records)} cards "
              f"(total: {len(all_records)}) | credits left: {remaining}")

        # save progress incrementally every few sets
        if i % 5 == 0:
            with open("price_history_dataset.json", "w") as f:
                json.dump(all_records, f)

        if len(all_records) >= CARD_TARGET:
            print(f"\nReached target of ~{CARD_TARGET} cards — stopping.")
            break

        time.sleep(0.3)

    with open("price_history_dataset.json", "w") as f:
        json.dump(all_records, f)
    print(f"\nDone. Saved {len(all_records)} cards with NM price history to price_history_dataset.json")

Found 219 sets. Pulling cards with 180 days of history...

[2/219] ME: 30th Celebration: +5 cards (total: 5) | credits left: 19909
[4/219] ME05: Pitch Black: +125 cards (total: 130) | credits left: 19659
[5/219] ME04: Chaos Rising: +127 cards (total: 257) | credits left: 19405
[6/219] First Partner Collection 2026: +3 cards (total: 260) | credits left: 19399
[7/219] ME03: Perfect Order: +131 cards (total: 391) | credits left: 19137
  Rate limited (429) — waiting 60s before retrying...
[8/219] ME: Ascended Heroes: +580 cards (total: 971) | credits left: 17976
[10/219] ME02: Phantasmal Flames: +138 cards (total: 1109) | credits left: 17698
  Rate limited (429) — waiting 60s before retrying...
[11/219] ME01: Mega Evolution: +194 cards (total: 1303) | credits left: 17309
[12/219] ME: Mega Evolution Promo: +116 cards (total: 1419) | credits left: 17033
[13/219] MEE: Mega Evolution Energies: +8 cards (total: 1427) | credits left: 17001
  Rate limited (429) — waiting 60s before retrying...
[1

In [1]:
# ===========================================================================
# POKEMON PREDICTOR — Multi-horizon training + prediction.
# Trains a SEPARATE model for each horizon (15/30/60/90 days), each a real
# forecast, and writes ONE card_predictions.json with all four per card.
# Also prints honest cross-validated accuracy for each horizon.
# ===========================================================================

import json
import numpy as np
import pickle
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False

HORIZONS = [15, 30, 60, 90]

# Quality filters (same liquid-card approach that gave the working model)
MIN_HISTORY = 160
MIN_PRICE = 8.00
MIN_AVG_VOLUME = 2
MAX_VOLATILITY = 0.6
SPARK_POINTS = 30
N_FOLDS = 5

with open("price_history_dataset.json") as f:
    raw = json.load(f)
print(f"Total cards in dataset: {len(raw)}")

def clean(card):
    h = [p for p in card["price_history"] if p["price"] is not None]
    h.sort(key=lambda p: p["date"])
    if len(h) < MIN_HISTORY:
        return None
    prices = [p["price"] for p in h]
    if np.median(prices) < MIN_PRICE:
        return None
    volumes = [(p.get("volume") or 0) for p in h]
    if np.mean(volumes) < MIN_AVG_VOLUME:
        return None
    cv = np.std(prices) / (np.mean(prices) + 1e-9)
    if cv > MAX_VOLATILITY:
        return None
    return prices, volumes

# ---- Build the TRAINING set (strict, liquid cards) ----
train_cards = []
for card in raw:
    r = clean(card)
    if r:
        prices, volumes = r
        train_cards.append({"rarity": card.get("rarity") or "Unknown",
                            "prices": prices, "volumes": volumes, "full_length": len(prices)})
print(f"Training cards (liquid, quality): {len(train_cards)}")

all_rarities = sorted(set(c["rarity"] for c in train_cards))
rarity_to_num = {r: i for i, r in enumerate(all_rarities)}

def pct(a, b): return 0.0 if a == 0 else (b - a) / a * 100

def features(prices, volumes, cutoff, rarity, full_length):
    wp, wv = prices[:cutoff], volumes[:cutoff]
    if len(wp) < 60:
        return None
    cur = wp[-1]; p7, p30, p60 = wp[-7], wp[-30], wp[-60]
    vol = np.std(wp[-30:]) / (np.mean(wp[-30:]) + 1e-9)
    arv = np.mean(wv[-30:]) if wv[-30:] else 0
    aov = np.mean(wv[-60:-30]) if wv[-60:-30] else 0
    vt = pct(aov + 1, arv + 1)
    acc = pct(p7, cur) - pct(p30, p7)
    lo, hi = min(wp), max(wp)
    pos = (cur - lo) / (hi - lo + 1e-9)
    drift = pct(wp[0], cur)
    # rarity may be unseen at prediction time (training used a strict subset);
    # fall back to a default index instead of crashing on unknown rarities.
    rarity_idx = rarity_to_num.get(rarity, len(rarity_to_num))
    return [pct(p7,cur), pct(p30,cur), pct(p60,cur), vol*100, np.log(cur+1),
            arv, vt, acc, rarity_idx, full_length, pos, drift]

def make_model():
    if HAVE_XGB:
        return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03,
                            subsample=0.8, colsample_bytree=0.8)
    return GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.03)

# ---- Train one model per horizon, report CV accuracy, save each ----
models = {}
accuracy = {}
for H in HORIZONS:
    X, y = [], []
    for c in train_cards:
        prices, volumes = c["prices"], c["volumes"]
        cutoff = len(prices) - H
        if cutoff < 60:
            continue
        f = features(prices, volumes, cutoff, c["rarity"], c["full_length"])
        if f is None:
            continue
        X.append(f)
        y.append(pct(prices[cutoff-1], prices[cutoff-1+H]))
    X, y = np.array(X), np.array(y)

    # cross-validated accuracy (honest)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    r2s = []
    for tr, te in kf.split(X):
        m = make_model(); m.fit(X[tr], y[tr])
        r2s.append(r2_score(y[te], m.predict(X[te])))
    r2s = np.array(r2s)
    accuracy[H] = round(float(r2s.mean()), 3)

    final = make_model(); final.fit(X, y)
    models[H] = final
    print(f"  {H}-day: trained on {len(X)} examples | CV R^2 = {r2s.mean():.3f} +/- {r2s.std():.3f}")

print(f"\nAccuracy by horizon: {accuracy}")

# ---- Generate predictions for ALL cards (lenient filter, full catalog) ----
PRED_MIN_HISTORY = 150
PRED_MIN_PRICE = 1.00

def downsample(prices, n):
    if len(prices) <= n:
        return [round(p, 2) for p in prices]
    idx = np.linspace(0, len(prices)-1, n).astype(int)
    return [round(prices[i], 2) for i in idx]

predictions = []
for card in raw:
    h = [p for p in card["price_history"] if p["price"] is not None]
    h.sort(key=lambda p: p["date"])
    if len(h) < PRED_MIN_HISTORY:
        continue
    prices = [p["price"] for p in h]
    if np.median(prices) < PRED_MIN_PRICE:
        continue
    volumes = [(p.get("volume") or 0) for p in h]
    rarity = card.get("rarity") or "Unknown"
    full_length = len(prices)

    f = features(prices, volumes, len(prices), rarity, full_length)
    if f is None:
        continue
    fx = np.array([f])
    current = prices[-1]

    horizon_data = {}
    for H in HORIZONS:
        change = float(models[H].predict(fx)[0])
        horizon_data[str(H)] = {
            "change_pct": round(change, 1),
            "price": round(current * (1 + change/100), 2),
        }

    predictions.append({
        "name": card["name"], "set": card["setName"],
        "rarity": rarity, "type": card.get("pokemonType") or None,
        "image": card.get("imageCdnUrl200") or card.get("imageUrl"),
        "image_large": card.get("imageCdnUrl400") or card.get("imageCdnUrl200") or card.get("imageUrl"),
        "tcgplayer_url": card.get("tcgPlayerUrl"),
        "current_price": round(current, 2),
        "horizons": horizon_data,
        # keep 60-day as the default top-level fields for backward compatibility
        "predicted_change_pct": horizon_data["60"]["change_pct"],
        "predicted_price": horizon_data["60"]["price"],
        "spark": downsample(prices, SPARK_POINTS),
        "full_history": downsample(prices, 90),
    })

# default-horizon summary (60-day) for the stat strip
changes60 = [p["horizons"]["60"]["change_pct"] for p in predictions]
summary = {
    "total_cards": len(predictions),
    "avg_change": round(float(np.mean(changes60)), 1),
    "predicted_rises": sum(1 for c in changes60 if c >= 5),
    "predicted_drops": sum(1 for c in changes60 if c <= -5),
    "horizons": HORIZONS,
    "accuracy_by_horizon": accuracy,
}

with open("card_predictions.json", "w") as f:
    json.dump({"summary": summary, "predictions": predictions}, f)

print(f"\nGenerated {len(predictions)} cards with 4 horizons each.")
print("Saved card_predictions.json")

Total cards in dataset: 9219
Training cards (liquid, quality): 814
  15-day: trained on 814 examples | CV R^2 = 0.021 +/- 0.117
  30-day: trained on 814 examples | CV R^2 = 0.129 +/- 0.045
  60-day: trained on 814 examples | CV R^2 = 0.184 +/- 0.084
  90-day: trained on 814 examples | CV R^2 = 0.424 +/- 0.046

Accuracy by horizon: {15: 0.021, 30: 0.129, 60: 0.184, 90: 0.424}

Generated 2985 cards with 4 horizons each.
Saved card_predictions.json
